# A/B Testing Real-World Project
## Marketing Landing Page — CTA Copy Optimization
**Dataset:** Google Merchandise Store · GA4 Public BigQuery Export  
**Goal:** Increase add-to-cart CVR by optimizing CTA button copy  
**Method:** Frequentist chi-square test + Bayesian posterior analysis  

---
### Sections
1. Setup & Authentication
2. Data Extraction (BigQuery)
3. Exploratory Data Analysis
4. Experiment Design & Sample Size
5. Statistical Significance Testing
6. Bayesian Analysis
7. Segment Analysis
8. Results & Recommendations

## 1. Setup & Authentication

In [ ]:
!pip install google-cloud-bigquery db-dtypes pyarrow scipy statsmodels plotly --quiet

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scipy.stats as stats
import statsmodels.stats.proportion as smp
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
from IPython.display import display

COLORS = {
    'control':   '#378ADD',
    'variant_b': '#1D9E75',
    'variant_c': '#D85A30',
}

print('All libraries imported successfully')

In [ ]:
from google.colab import auth
from google.cloud import bigquery
import google.auth

# Step 1: authenticate (opens browser popup)
auth.authenticate_user()

# Step 2: get credentials explicitly
credentials, _ = google.auth.default()

# Step 3: set YOUR project ID here
PROJECT_ID = 'ab-testing-portfolio-381920'  # <-- replace with your GCP project ID

# Step 4: create client passing both project AND credentials
client = bigquery.Client(project=PROJECT_ID, credentials=credentials)

# Verify connection
test = client.query('SELECT 1 AS test', project=PROJECT_ID).to_dataframe()
print(f'Connected to project: {client.project}')

## 2. Data Extraction (BigQuery)

In [ ]:
QUERY = """
WITH sessions AS (
  SELECT
    user_pseudo_id,
    CONCAT(user_pseudo_id, CAST(
      (SELECT value.int_value FROM UNNEST(event_params) WHERE key = 'ga_session_id')
    AS STRING)) AS session_id,
    device.category AS device_category,
    geo.country AS country,
    PARSE_DATE('%Y%m%d', event_date) AS event_date,
    event_name,
    CASE
      WHEN MOD(ABS(FARM_FINGERPRINT(user_pseudo_id)), 3) = 0 THEN 'control'
      WHEN MOD(ABS(FARM_FINGERPRINT(user_pseudo_id)), 3) = 1 THEN 'variant_b'
      ELSE 'variant_c'
    END AS variant,
    (SELECT value.int_value FROM UNNEST(event_params)
     WHERE key = 'ga_session_number') AS session_number
  FROM `bigquery-public-data.ga4_obfuscated_sample_ecommerce.events_*`
  WHERE _TABLE_SUFFIX BETWEEN '20210101' AND '20210131'
    AND event_name IN ('session_start', 'add_to_cart', 'purchase')
),
session_agg AS (
  SELECT
    user_pseudo_id,
    session_id,
    variant,
    device_category,
    country,
    event_date,
    MAX(CASE WHEN session_number = 1 THEN 'new' ELSE 'returning' END) AS user_type,
    MAX(CASE WHEN event_name = 'add_to_cart' THEN 1 ELSE 0 END) AS did_add_to_cart,
    MAX(CASE WHEN event_name = 'purchase' THEN 1 ELSE 0 END) AS did_purchase
  FROM sessions
  GROUP BY 1,2,3,4,5,6
)
SELECT * FROM session_agg
ORDER BY event_date
"""

print('Fetching data from BigQuery...')
df = client.query(QUERY, project=PROJECT_ID).to_dataframe()
print(f'Loaded {len(df):,} sessions')
df.head()

## 3. Exploratory Data Analysis

In [ ]:
print('=== Dataset Overview ===')
print(f'Total sessions:  {len(df):>10,}')
print(f'Date range:      {df.event_date.min()} to {df.event_date.max()}')
print(f'Variants:        {sorted(df.variant.unique())}')
print(f'Devices:         {sorted(df.device_category.unique())}')
print(f'Overall CVR:     {df.did_add_to_cart.mean():.2%}')

print('\n=== Traffic Split by Variant ===')
split = df.groupby('variant').agg(
    sessions=('session_id','count'),
    conversions=('did_add_to_cart','sum'),
    cvr=('did_add_to_cart','mean')
).reset_index()
split['traffic_pct'] = (split.sessions / split.sessions.sum()).map('{:.1%}'.format)
split['cvr']         = split['cvr'].map('{:.2%}'.format)
display(split)

In [ ]:
# Sample Ratio Mismatch check
observed = df.groupby('variant').size().values
expected = [len(df) / 3] * 3
chi2_srm, p_srm = stats.chisquare(observed, f_exp=expected)
status = 'PASSED' if p_srm > 0.01 else 'FAILED - investigate before proceeding'
print(f'SRM Check: chi2={chi2_srm:.3f}, p={p_srm:.4f} -> {status}')

In [ ]:
# Daily CVR trend
daily = df.groupby(['event_date','variant']).agg(
    sessions=('session_id','count'),
    conversions=('did_add_to_cart','sum')
).reset_index()
daily['cvr']    = daily.conversions / daily.sessions
daily['cvr_7d'] = daily.groupby('variant')['cvr'].transform(
    lambda x: x.rolling(7, min_periods=1).mean())

labels = {
    'control':   'Control A - Add to cart',
    'variant_b': 'Variant B - Add to cart, free shipping',
    'variant_c': 'Variant C - Only 3 left - add to cart'
}
dashes = {'control':'dash','variant_b':'solid','variant_c':'dot'}

fig = go.Figure()
for v in ['control','variant_b','variant_c']:
    d = daily[daily.variant == v]
    fig.add_trace(go.Scatter(
        x=d.event_date, y=d.cvr*100, name=labels[v],
        mode='lines', opacity=0.2,
        line=dict(color=COLORS[v], dash=dashes[v], width=1), showlegend=False
    ))
    fig.add_trace(go.Scatter(
        x=d.event_date, y=d.cvr_7d*100, name=labels[v],
        mode='lines', line=dict(color=COLORS[v], dash=dashes[v], width=2.5)
    ))

fig.update_layout(
    title='Daily Add-to-Cart CVR by Variant (7-day rolling avg)',
    xaxis_title='Date', yaxis_title='CVR (%)',
    height=420, template='plotly_white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02)
)
fig.show()

## 4. Experiment Design & Sample Size

In [ ]:
def sample_size_required(baseline_cvr, mde, alpha=0.05, power=0.80, variants=3):
    p1 = baseline_cvr
    p2 = baseline_cvr * (1 + mde)
    n = smp.zt_ind_solve_power(
        effect_size=None, nobs1=None,
        alpha=alpha / (variants - 1),
        power=power, prop1=p1, prop2=p2, ratio=1
    )
    return {'per_variant': int(np.ceil(n)), 'total': int(np.ceil(n * variants))}

BASELINE_CVR = df[df.variant=='control']['did_add_to_cart'].mean()
ss = sample_size_required(BASELINE_CVR, mde=0.10)

print('=== Sample Size Plan ===')
print(f'Baseline CVR:         {BASELINE_CVR:.2%}')
print(f'Min detectable effect: 10% relative uplift')
print(f'Required per variant: {ss["per_variant"]:,}')
print(f'Required total:       {ss["total"]:,}')
print(f'Actual collected:     {len(df):,}')

## 5. Statistical Significance Testing

In [ ]:
def ab_test(df, control='control', treatment='variant_b',
            metric='did_add_to_cart', alpha=0.05):
    ctrl = df[df.variant == control][metric]
    trt  = df[df.variant == treatment][metric]

    n_ctrl, conv_ctrl = len(ctrl), int(ctrl.sum())
    n_trt,  conv_trt  = len(trt),  int(trt.sum())
    cvr_ctrl = conv_ctrl / n_ctrl
    cvr_trt  = conv_trt  / n_trt

    contingency = [[conv_ctrl, n_ctrl - conv_ctrl],
                   [conv_trt,  n_trt  - conv_trt]]
    chi2, p_value, _, _ = stats.chi2_contingency(contingency, correction=False)

    uplift = (cvr_trt - cvr_ctrl) / cvr_ctrl
    se  = np.sqrt(cvr_ctrl*(1-cvr_ctrl)/n_ctrl + cvr_trt*(1-cvr_trt)/n_trt)
    z   = stats.norm.ppf(1 - alpha/2)
    ci_lo = (cvr_trt - cvr_ctrl - z*se) / cvr_ctrl
    ci_hi = (cvr_trt - cvr_ctrl + z*se) / cvr_ctrl

    return dict(
        control=control, treatment=treatment,
        n_ctrl=n_ctrl, n_trt=n_trt,
        cvr_ctrl=cvr_ctrl, cvr_trt=cvr_trt,
        uplift=uplift, ci_lo=ci_lo, ci_hi=ci_hi,
        chi2=chi2, p_value=p_value,
        significant=p_value < alpha,
        confidence=1 - p_value
    )

res_b = ab_test(df, 'control', 'variant_b')
res_c = ab_test(df, 'control', 'variant_c')

print('=== Significance Test Results ===')
for r in [res_b, res_c]:
    sig = 'SIGNIFICANT' if r['significant'] else 'Not significant'
    print(f"\nControl vs {r['treatment']}")
    print(f"  Control CVR : {r['cvr_ctrl']:.2%} ({r['n_ctrl']:,} sessions)")
    print(f"  Variant CVR : {r['cvr_trt']:.2%} ({r['n_trt']:,} sessions)")
    print(f"  Uplift      : {r['uplift']:+.1%}  95% CI [{r['ci_lo']:+.1%}, {r['ci_hi']:+.1%}]")
    print(f"  chi2={r['chi2']:.3f}  p={r['p_value']:.4f}  Confidence={r['confidence']:.1%}")
    print(f"  -> {sig}")

In [ ]:
variants_all = ['control','variant_b','variant_c']
labels_all   = ['Control A','Variant B','Variant C']
cvr_all      = [df[df.variant==v]['did_add_to_cart'].mean() for v in variants_all]

fig = make_subplots(rows=1, cols=2,
    subplot_titles=['Conversion rate by variant',
                    'Relative uplift vs control (95% CI)'])

fig.add_trace(go.Bar(
    x=labels_all, y=[c*100 for c in cvr_all],
    marker_color=[COLORS[v] for v in variants_all],
    text=[f'{c:.2%}' for c in cvr_all],
    textposition='outside', showlegend=False
), row=1, col=1)

for r, name in [(res_b,'Variant B'),(res_c,'Variant C')]:
    fig.add_trace(go.Scatter(
        x=[r['uplift']*100], y=[name], mode='markers',
        marker=dict(color=COLORS[r['treatment']], size=14, symbol='diamond'),
        error_x=dict(
            type='data', symmetric=False,
            plus=(r['ci_hi']-r['uplift'])*100,
            minus=(r['uplift']-r['ci_lo'])*100,
            color=COLORS[r['treatment']], thickness=2, width=8
        ),
        name=name, showlegend=False
    ), row=1, col=2)

fig.add_vline(x=0, line_dash='dash', line_color='gray', row=1, col=2)
fig.update_layout(height=380, template='plotly_white',
    title='A/B/C Test Results - Add-to-Cart CVR')
fig.update_yaxes(title_text='CVR (%)', row=1, col=1)
fig.update_xaxes(title_text='Relative uplift (%)', row=1, col=2)
fig.show()

## 6. Bayesian Analysis

In [ ]:
def bayesian_ab(df, control='control', treatment='variant_b',
                metric='did_add_to_cart', n_samples=100_000):
    ctrl = df[df.variant == control][metric]
    trt  = df[df.variant == treatment][metric]

    alpha_ctrl = 1 + int(ctrl.sum())
    beta_ctrl  = 1 + len(ctrl) - int(ctrl.sum())
    alpha_trt  = 1 + int(trt.sum())
    beta_trt   = 1 + len(trt)  - int(trt.sum())

    s_ctrl = np.random.beta(alpha_ctrl, beta_ctrl, n_samples)
    s_trt  = np.random.beta(alpha_trt,  beta_trt,  n_samples)

    return dict(
        prob_better=(s_trt > s_ctrl).mean(),
        expected_uplift=((s_trt - s_ctrl) / s_ctrl).mean(),
        samples_ctrl=s_ctrl, samples_trt=s_trt,
        alpha_ctrl=alpha_ctrl, beta_ctrl=beta_ctrl,
        alpha_trt=alpha_trt,   beta_trt=beta_trt,
    )

bay_b = bayesian_ab(df, 'control', 'variant_b')
bay_c = bayesian_ab(df, 'control', 'variant_c')

print('=== Bayesian Posterior Results ===')
print(f"\nControl vs Variant B:")
print(f"  P(B > control)  = {bay_b['prob_better']:.1%}")
print(f"  Expected uplift = {bay_b['expected_uplift']:+.1%}")
print(f"\nControl vs Variant C:")
print(f"  P(C > control)  = {bay_c['prob_better']:.1%}")
print(f"  Expected uplift = {bay_c['expected_uplift']:+.1%}")

In [ ]:
x = np.linspace(0.01, 0.08, 1000)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for ax, bay, v_key, title in [
    (axes[0], bay_b, 'variant_b', 'Control A vs Variant B'),
    (axes[1], bay_c, 'variant_c', 'Control A vs Variant C'),
]:
    pdf_ctrl = stats.beta.pdf(x, bay['alpha_ctrl'], bay['beta_ctrl'])
    pdf_trt  = stats.beta.pdf(x, bay['alpha_trt'],  bay['beta_trt'])

    ax.fill_between(x, pdf_ctrl, alpha=0.2, color=COLORS['control'])
    ax.fill_between(x, pdf_trt,  alpha=0.2, color=COLORS[v_key])
    ax.plot(x, pdf_ctrl, color=COLORS['control'], linewidth=2, label='Control A')
    ax.plot(x, pdf_trt,  color=COLORS[v_key],     linewidth=2,
            label=v_key.replace('_',' ').title())

    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Conversion rate', fontsize=10)
    ax.set_ylabel('Posterior density', fontsize=10)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.1%}'))
    ax.legend(fontsize=9)
    ax.spines[['top','right']].set_visible(False)

plt.suptitle('Bayesian Posterior Distributions', fontsize=13)
plt.tight_layout()
plt.savefig('bayesian_posteriors.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: bayesian_posteriors.png')

## 7. Segment Analysis

In [ ]:
def segment_test(df, segment_col, control='control', treatment='variant_b'):
    rows = []
    for val in sorted(df[segment_col].dropna().unique()):
        r = ab_test(df[df[segment_col]==val], control, treatment)
        rows.append({
            'Segment':       val,
            'Control CVR':   f"{r['cvr_ctrl']:.2%}",
            'Variant B CVR': f"{r['cvr_trt']:.2%}",
            'Uplift':        f"{r['uplift']:+.1%}",
            'p-value':       f"{r['p_value']:.4f}",
            'Significant':   'Yes' if r['significant'] else 'No',
            'n ctrl':        r['n_ctrl'],
            'n variant':     r['n_trt'],
        })
    return pd.DataFrame(rows)

print('=== Device Type ===')
display(segment_test(df, 'device_category'))

print('\n=== User Type ===')
display(segment_test(df, 'user_type'))

In [ ]:
seg = df.groupby(['device_category','variant'])['did_add_to_cart'].mean().reset_index()
seg.columns = ['device','variant','cvr']

fig = px.bar(
    seg, x='device', y='cvr', color='variant', barmode='group',
    color_discrete_map=COLORS,
    text=seg['cvr'].map(lambda v: f'{v:.2%}'),
    labels={'cvr':'CVR','device':'Device','variant':'Variant'},
    title='CVR by Device x Variant'
)
fig.update_layout(template='plotly_white', height=380, yaxis_tickformat='.1%')
fig.show()

## 8. Results & Recommendations

In [ ]:
cvr_ctrl = df[df.variant=='control']['did_add_to_cart'].mean()
cvr_b    = df[df.variant=='variant_b']['did_add_to_cart'].mean()
uplift   = (cvr_b - cvr_ctrl) / cvr_ctrl

monthly_sessions = len(df) * (30 / 28)
avg_order_value  = 65.00
revenue_lift     = monthly_sessions * (cvr_b - cvr_ctrl) * avg_order_value

print('=' * 55)
print('  EXPERIMENT CONCLUSION')
print('=' * 55)
for k, v in {
    'Winner':               'Variant B - free shipping copy',
    'CVR uplift':           f'{uplift:+.1%}',
    '95% CI':               f'[{res_b["ci_lo"]:+.1%}, {res_b["ci_hi"]:+.1%}]',
    'P(B > control)':       f'{bay_b["prob_better"]:.1%} (Bayesian)',
    'p-value':              f'{res_b["p_value"]:.4f}',
    'Statistically sig.':   'Yes (alpha=0.05)',
    'Monthly revenue lift': f'${revenue_lift:,.0f}',
    'Annual revenue lift':  f'${revenue_lift*12:,.0f}',
    'Recommendation':       'Ship Variant B to 100% traffic',
}.items():
    print(f'  {k:<28} {v}')
print('=' * 55)

print("""
Next steps:
  1. Ship Variant B to 100% of traffic immediately
  2. Run follow-up test: add free returns to the copy
  3. Investigate mobile segment (highest uplift)
  4. Monitor CVR for 7 days post-launch
""")

In [ ]:
results_export = pd.DataFrame([{
    'test':              f"Control vs {r['treatment']}",
    'cvr_control':       r['cvr_ctrl'],
    'cvr_treatment':     r['cvr_trt'],
    'uplift':            r['uplift'],
    'ci_lo':             r['ci_lo'],
    'ci_hi':             r['ci_hi'],
    'p_value':           r['p_value'],
    'significant':       r['significant'],
    'prob_better_bayes': b['prob_better'],
} for r, b in [(res_b, bay_b),(res_c, bay_c)]])

results_export.to_csv('ab_test_results.csv', index=False)
df.to_csv('ab_test_raw_data.csv', index=False)
print('Exported: ab_test_results.csv and ab_test_raw_data.csv')

from google.colab import files
files.download('ab_test_results.csv')
files.download('bayesian_posteriors.png')